# 04 - Final Predictions Scratch Validation

Stage 4 scratch validation only: compute real Elo ratings, run the Monte Carlo bracket simulator, and inspect tournament-win/final-reach probabilities. This is not the final submission pipeline yet.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import DATA_RAW, N_SIMS
from src.bracket import monte_carlo_bracket
from src.data_loader import clean_results, fetch_international_results, load_fixtures
from src.elo import compute_elo_ratings

In [2]:
raw_results = fetch_international_results(force_refresh=False)
results = clean_results(raw_results)
ratings, history = compute_elo_ratings(results)

group_fixtures, knockout_slots = load_fixtures(
    DATA_RAW / "group_fixtures.csv",
    DATA_RAW / "knockout_slots.csv",
)

advancement, modal_slots = monte_carlo_bracket(
    group_fixtures,
    knockout_slots,
    ratings,
    n_sims=N_SIMS,
)

print(f"Simulations: {N_SIMS:,}")
print(f"Modal knockout slots: {len(modal_slots)}")

Simulations: 10,000
Modal knockout slots: 64


In [3]:
top_15_win = advancement.sort_values("p_win_tournament", ascending=False).head(15)
top_15_win[["team", "p_win_tournament", "p_reach_final", "p_reach_SF", "p_reach_QF"]]

,team,p_win_tournament,p_reach_final,p_reach_SF,p_reach_QF
0,Spain,0.2570,0.3666,0.4859,0.5968
1,Argentina,0.1659,0.2677,0.3909,0.5297
2,France,0.0827,0.1501,0.2653,0.4155
3,England,0.0574,0.1137,0.2028,0.3416
4,Mexico,0.0440,0.1017,0.2128,0.4310
5,Morocco,0.0429,0.0891,0.1893,0.3206
6,Brazil,0.0415,0.0897,0.1890,0.3213
7,Portugal,0.0388,0.0815,0.1627,0.3042
8,Japan,0.0369,0.0804,0.1648,0.2945
9,Germany,0.0251,0.0587,0.1428,0.2827


In [4]:
top_10_final = advancement.sort_values("p_win_tournament", ascending=False).head(10)
top_10_final[["team", "p_reach_final"]]

,team,p_reach_final
0,Spain,0.3666
1,Argentina,0.2677
2,France,0.1501
3,England,0.1137
4,Mexico,0.1017
5,Morocco,0.0891
6,Brazil,0.0897
7,Portugal,0.0815
8,Japan,0.0804
9,Germany,0.0587


## Sanity Note

The validation read should put the highest-rated favorites near the top of tournament-win probability, while still leaving meaningful probability mass for bracket path difficulty and upsets. Spain, Argentina, France, and Brazil should be prominent; if a weak team dominates or the field is almost flat, revisit Elo calibration, fixture resolution, or the Poisson spread before building `predict.py`.

In [5]:
from src.poisson_model import team_lambdas, score_matrix, outcome_probs
# Mexico (host, 1803) vs an even-rated NON-host team — does the host bump alone make them a big favorite?
for nm, rb in [("vs equal non-host", 1803), ("vs France 1858", 1858)]:
    la, lb = team_lambdas(1803, rb, team_a="Mexico", team_b="X", neutral=True)
    ph,pd,pa = outcome_probs(score_matrix(la,lb))
    print(f"Mexico {nm}: λ {la:.2f}-{lb:.2f}  W/D/L {ph:.0%}/{pd:.0%}/{pa:.0%}")

Mexico vs equal non-host: λ 1.73-1.35  W/D/L 47%/23%/30%
Mexico vs France 1858: λ 1.52-1.54  W/D/L 38%/24%/38%


In [6]:
# crude but illustrative: implied per-round survival from the advancement table
mex = advancement.set_index("team").loc["Mexico"]
print("reach R16  :", round(mex["p_reach_R16"], 3) if "p_reach_R16" in mex else "n/a")
print("reach QF   :", round(mex["p_reach_QF"], 3))
print("reach SF   :", round(mex["p_reach_SF"], 3))
print("reach final:", round(mex["p_reach_final"], 3))
print("win        :", round(mex["p_win_tournament"], 3))
# ratio of consecutive rounds ≈ conditional win prob in that round
print("\nimplied conditional win % each round:")
print("QF->SF :", round(mex["p_reach_SF"]/mex["p_reach_QF"], 2))
print("SF->F  :", round(mex["p_reach_final"]/mex["p_reach_SF"], 2))
print("F->win :", round(mex["p_win_tournament"]/mex["p_reach_final"], 2))

reach R16  : 0.725
reach QF   : 0.479
reach SF   : 0.288
reach final: 0.156
win        : 0.08

implied conditional win % each round:
QF->SF : 0.6
SF->F  : 0.54
F->win : 0.51


In [6]:
import pandas as pd
ko = pd.read_csv("D:\\datacamp competetion\\wc2026-forecasting\\outputs\\knockout_predictions.csv")
# show Germany's predicted path
g = ko[(ko.predicted_home_team=="Germany") | (ko.predicted_away_team=="Germany")]
print(g[["match_id","round","predicted_home_team","predicted_away_team","match_winner"]].to_string(index=False))

 match_id         round predicted_home_team predicted_away_team match_winner
       75   Round of 32             Germany               Qatar         home
       77   Round of 32             Germany             Senegal         away
       89   Round of 16         South Korea             Germany         away
       97 Quarter-final             Germany              Brazil         away
      101    Semi-final             Germany               Spain         away


In [8]:
import numpy as np
from src.poisson_model import team_lambdas, score_matrix
from src.optimizer import best_score
from src import scoring

la, lb = team_lambdas(1900, 1600, neutral=True)
m = score_matrix(la, lb)
ev_pick = tuple(int(x) for x in best_score(m, "any"))
modal   = tuple(int(x) for x in np.unravel_index(np.argmax(m), m.shape))

def score_tiers_only(pred, true):   # 25/10/10 ONLY, no winner bonus
    if pred == true: return 25
    if pred[0]-pred[1] == true[0]-true[1]: return 10
    if pred[0]+pred[1] == true[0]+true[1]: return 10
    return 0
def exp_pts(pred):
    return sum(m[i,j]*score_tiers_only(pred,(i,j)) for i in range(m.shape[0]) for j in range(m.shape[1]))

print("EV pick:", ev_pick, "score-only EV pts:", round(exp_pts(ev_pick),3))
print("Modal  :", modal,   "score-only EV pts:", round(exp_pts(modal),3))

EV pick: (2, 0) score-only EV pts: 4.734
Modal  : (2, 0) score-only EV pts: 4.734


In [15]:
import pandas as pd
from pathlib import Path

# walk up from the current dir until we find the repo root (the folder containing 'data')
p = Path.cwd()
while not (p / "data" / "processed" / "feature_matrix.parquet").exists() and p != p.parent:
    p = p.parent

fm_path = p / "data" / "processed" / "feature_matrix.parquet"
print("cwd:", Path.cwd())
print("found:", fm_path, "->", fm_path.exists())

fm = pd.read_parquet(fm_path)
print(fm.shape)

cwd: d:\datacamp competetion\wc2026-forecasting\notebooks
found: d:\datacamp competetion\wc2026-forecasting\data\processed\feature_matrix.parquet -> True
(25256, 53)


In [17]:
# 1. confirm all feature families + labels are present
print(fm.columns.tolist())

['date', 'home_team', 'away_team', 'tournament', 'neutral', 'home_goals', 'away_goals', 'outcome', 'home_elo', 'away_elo', 'elo_diff', 'tournament_tier', 'is_major_tournament', 'is_qualifier', 'is_friendly', 'home_is_host', 'away_is_host', 'home_prior_matches', 'home_has_min_history', 'home_unbeaten_streak', 'home_winning_streak', 'home_clean_sheet_rate_l5', 'home_draw_rate_l10', 'home_major_prior_matches', 'home_major_win_rate', 'home_last5_ppg', 'home_last5_gf_avg', 'home_last5_ga_avg', 'home_last5_win_rate', 'home_last10_ppg', 'home_last10_gf_avg', 'home_last10_ga_avg', 'home_last10_win_rate', 'away_prior_matches', 'away_has_min_history', 'away_unbeaten_streak', 'away_winning_streak', 'away_clean_sheet_rate_l5', 'away_draw_rate_l10', 'away_major_prior_matches', 'away_major_win_rate', 'away_last5_ppg', 'away_last5_gf_avg', 'away_last5_ga_avg', 'away_last5_win_rate', 'away_last10_ppg', 'away_last10_gf_avg', 'away_last10_ga_avg', 'away_last10_win_rate', 'h2h_count', 'h2h_home_win_rate'

In [28]:
import sys
sys.path.insert(0, str(p))   # p = repo root, from the earlier cell
braz_rows = fm[fm.home_team=="Brazil"]
print("Brazil home matches in matrix:", len(braz_rows))
row = braz_rows.iloc[len(braz_rows)//2]   # pick the middle one, guaranteed to exist
print("Match:", row['date'], row['home_team'], "vs", row['away_team'])
print("stored last-5 PPG feature:", row.filter(like='form5').to_dict())

from src.data_loader import fetch_international_results, clean_results
res = clean_results(fetch_international_results())
res['date'] = pd.to_datetime(res['date'])
d = pd.Timestamp(row['date'])
braz = res[((res.home_team=="Brazil")|(res.away_team=="Brazil")) & (res.date < d)].sort_values('date').tail(5)
print("\nBrazil's actual prior 5 (must all be < match date):")
print(braz[['date','home_team','away_team','home_score','away_score']].to_string(index=False))
pts = sum((3 if (gf:=(m.home_score if m.home_team=="Brazil" else m.away_score))>(ga:=(m.away_score if m.home_team=="Brazil" else m.home_score)) else (1 if gf==ga else 0)) for _,m in braz.iterrows())
print("hand-computed last-5 PPG:", round(pts/len(braz),3))

Brazil home matches in matrix: 193
Match: 2013-06-22 00:00:00 Brazil vs Italy
stored last-5 PPG feature: {}

Brazil's actual prior 5 (must all be < match date):
      date home_team away_team  home_score  away_score
2013-04-24    Brazil     Chile           2           2
2013-06-02    Brazil   England           2           2
2013-06-09    Brazil    France           3           0
2013-06-15    Brazil     Japan           3           0
2013-06-19    Brazil    Mexico           2           0
hand-computed last-5 PPG: 2.2


In [29]:
print("stored home_last5_ppg:", row['home_last5_ppg'])

stored home_last5_ppg: 2.2
